# calculating cost and time statistics
this notebook is the clean version

In [ ]:
from pathlib import Path
import sys
import os

current_directory = Path(os.getcwd())
print(current_directory)
root_directory = current_directory.parent.parent

sys.path.append(str(root_directory))
import typing as t

In [ ]:
# the final part of this path is the relative path to the evaluation directory from the root of the project
EVAL_DIR = current_directory / "../.." / "data/evaluation/regenerate_from_failure_point/test_with_hammer-cde7b07affd2400dbcc01fb8e13b376e"

In [ ]:
from src.dataset import COQ_WIGDERSON_TEST_SAMPLED_DATASET, COQGYM_TEST_SAMPLED_DATASET
import json

from src.strategy.goal_decomposition import get_proof

DATASET = COQGYM_TEST_SAMPLED_DATASET


In [ ]:
def get_success(json: t.Dict[str, t.Any]) -> bool:
    """
    Returns true if an example's json file indicates that the example was successfully proven.
    This varies based on the evaluation. Currently, it is for regenerate_from_failure_point.
    """
    return json["debug_success"]

In [ ]:
successes: t.Set[str] = set()
for example in DATASET:
    example_file = EVAL_DIR / f"{example.name}.json"
    try:
        example_json = json.load(open(example_file))
        if get_success(example_json):
            successes.add(example.name)
    except FileNotFoundError:
        continue

print(len(successes), "/", len(DATASET))

In [ ]:
# a set of example names that were successfully proven with just a hammer.
# these will have LLM usage 0, since they did not use the LLM.
HAMMER_SUCCESSES = set([
])

# a set of example names that were successfully proven with a single LLM invocation.
# these have the max LLM usage of a single invocation. Often, they don't have logs because the
# process is completed before the logs are written.
ZERO_SHOT_SUCCESSES = set([
])


In [ ]:
ARGS = (
    EVAL_DIR / 'output.log',
    successes,
    HAMMER_SUCCESSES,
    ZERO_SHOT_SUCCESSES
)

In [ ]:
import json

class Stats(t.TypedDict):
    min: float
    max: float
    mean: float
    median: float


def median(lst):
    n = len(lst)
    if n % 2 == 1:
        return sorted(lst)[n//2]
    else:
        return sum(sorted(lst)[n//2-1:n//2+1]) / 2
    
def mean(lst):
    return sum(lst) / len(lst)

def stats(d: t.Dict[str, float]) -> Stats:
    values = d.values()
    return {
        "min": min(values),
        "max": max(values),
        "mean": mean(values),
        "median": median(values)
    }
    

## cost

In [ ]:
import re
def parse_usage_str(usage_str):
    pattern = r"Usage\(name='([^']+)',\s+model='([^']+)',\s+end_time=datetime\.datetime\(([^)]+)\),\s+duration_millis=(\d+),\s+num_input_tokens=(\d+),\s+num_output_tokens=(\d+),\s+num_tokens=(\d+),\s+num_requests=(\d+),\s+children=\[\],\s+uuid='([^']+)'\)"
    match = re.search(pattern, usage_str)
    if match:
        return {
            'name': match.group(1),
            'model': match.group(2),
            'end_time': match.group(3),
            'duration_millis': match.group(4),
            'num_input_tokens': match.group(5),
            'num_output_tokens': match.group(6),
            'num_tokens': match.group(7),
            'num_requests': match.group(8),
            'uuid': match.group(9)
        }
    else:
        return None
    
def sum_usage(usages: t.List[t.Dict[str, str]]) -> t.Dict[str, int]:
    return {
        'num_input_tokens': sum(int(u['num_input_tokens']) for u in usages),
        'num_output_tokens': sum(int(u['num_output_tokens']) for u in usages),
        'num_tokens': sum(int(u['num_tokens']) for u in usages),
        'num_requests': sum(int(u['num_requests']) for u in usages),
    }

INPUT_TOKEN_PRICE = 30 / 1_000_000
OUTPUT_TOKEN_PRICE = 60 / 1_000_000

def price(usage: t.Dict[str, int]) -> float:
    return usage['num_input_tokens'] * INPUT_TOKEN_PRICE + usage['num_output_tokens'] * OUTPUT_TOKEN_PRICE


def max_usage(usages: t.List[t.Dict[str, str]]) -> t.Dict[str, int]:
    return {
        'num_input_tokens': max(int(u['num_input_tokens']) for u in usages),
        'num_output_tokens': max(int(u['num_output_tokens']) for u in usages),
        'num_tokens': max(int(u['num_tokens']) for u in usages),
        'num_requests': max(int(u['num_requests']) for u in usages),
    }

In [ ]:
def calculate_example_to_usage(
    log_path: Path
) -> t.Tuple[t.Dict[str, t.List[t.Dict[str, str]]], t.Dict[str, t.Dict[str, int]]]:
    """
    Calculate usage statistics for each example from a log file.

    Args:
        log_path: Path to the log file
        successes: Set of successful example names
        hammer_successes: Set of examples solved by hammer
        zero_shot_successes: Set of examples solved by zero-shot
    """

    # Parse log file
    with open(log_path, 'r') as f:
        lines = f.readlines()
    lines_dicts = [json.loads(l) for l in lines]
    
    # Extract usage data
    usage = []
    for line in lines_dicts:
        if "llm.gpt" in line['name'] and 'usage' in line:
            usage.append((line['example_name'], line['usage'], line['run_uuid']))

    # Parse usage strings
    def parse_usage_str(usage_str: str) -> t.Optional[t.Dict[str, str]]:
        pattern = r"Usage\(name='([^']+)',\s+model='([^']+)',\s+end_time=datetime\.datetime\(([^)]+)\),\s+duration_millis=(\d+),\s+num_input_tokens=(\d+),\s+num_output_tokens=(\d+),\s+num_tokens=(\d+),\s+num_requests=(\d+),\s+children=\[\],\s+uuid='([^']+)'\)"
        import re
        match = re.search(pattern, usage_str)
        if match:
            return {
                'name': match.group(1),
                'model': match.group(2),
                'end_time': match.group(3),
                'duration_millis': match.group(4),
                'num_input_tokens': match.group(5),
                'num_output_tokens': match.group(6),
                'num_tokens': match.group(7),
                'num_requests': match.group(8),
                'uuid': match.group(9)
            }
        return None

    # Group usage by example
    example_to_parsed_usage = {}
    for example_name, usage_str, run_uuid in usage:
        parsed = parse_usage_str(usage_str)
        if parsed is None:
            continue
        if example_name not in example_to_parsed_usage:
            example_to_parsed_usage[example_name] = []
        example_to_parsed_usage[example_name].append(parsed)

    # Calculate total usage per example
    def sum_usage(usages: t.List[t.Dict[str, str]]) -> t.Dict[str, int]:
        return {
            'num_input_tokens': sum(int(u['num_input_tokens']) for u in usages),
            'num_output_tokens': sum(int(u['num_output_tokens']) for u in usages),
            'num_tokens': sum(int(u['num_tokens']) for u in usages),
            'num_requests': sum(int(u['num_requests']) for u in usages),
        }

    example_to_usage = {
        example_name: sum_usage(usages) 
        for example_name, usages in example_to_parsed_usage.items()
    }

    return example_to_parsed_usage, example_to_usage

def calculate_usage_stats(
    example_to_parsed_usage: t.Dict[str, t.List[t.Dict[str, str]]],
    example_to_usage: t.Dict[str, t.Dict[str, int]],
) -> t.Dict[str, Stats]:
    """
    Calculate usage statistics from parsed usage data.

    Args:
        example_to_parsed_usage: Dictionary mapping example names to parsed usage data
        example_to_usage: Dictionary mapping example names to usage data

    Returns:
        Dictionary mapping example names to usage statistics
        each example has a dictionary with the following keys:
        - num_input_tokens
        - num_output_tokens
        - num_tokens
        - num_requests

        each key has the following stats:
        - min
        - max
        - mean
        - median
    """

    example_to_input_tokens = {
        example_name: float(usage['num_input_tokens'])
        for example_name, usage in example_to_usage.items()
    }
    example_to_output_tokens = {
        example_name: float(usage['num_output_tokens'])
        for example_name, usage in example_to_usage.items()
    }
    example_to_tokens = {
        example_name: float(usage['num_tokens'])
        for example_name, usage in example_to_usage.items()
    }
    example_to_requests = {
        example_name: float(usage['num_requests'])
        for example_name, usage in example_to_usage.items()
    }

    example_to_usage_stats = {
        'num_input_tokens': stats(example_to_input_tokens),
        'num_output_tokens': stats(example_to_output_tokens),
        'num_tokens': stats(example_to_tokens),
        'num_requests': stats(example_to_requests)
    }

    return example_to_usage_stats
    
    
    

def calculate_cost_stats(
    example_to_parsed_usage: t.Dict[str, t.List[t.Dict[str, str]]],
    example_to_usage: t.Dict[str, t.Dict[str, int]],
    successes: t.Set[str],
    hammer_successes: t.Set[str],
    zero_shot_successes: t.Set[str],
    input_token_price: float = 30 / 1_000_000,
    output_token_price: float = 60 / 1_000_000
) -> t.Dict[str, Stats]:
    """
    Calculate cost statistics from log data.
    
    Args:
        log_path: Path to the log file
        successes: Set of successful example names
        hammer_successes: Set of examples solved by hammer
        zero_shot_successes: Set of examples solved by zero-shot
        input_token_price: Price per input token
        output_token_price: Price per output token
        
    Returns:
        Dictionary containing stats for different cost metrics
    """
    

    # Calculate price
    def price(usage: t.Dict[str, int]) -> float:
        return (usage['num_input_tokens'] * input_token_price + 
                usage['num_output_tokens'] * output_token_price)

    # Calculate max usage/price for corrections
    def max_usage(usages: t.List[t.Dict[str, str]]) -> t.Dict[str, int]:
        return {
            'num_input_tokens': max(int(u['num_input_tokens']) for u in usages),
            'num_output_tokens': max(int(u['num_output_tokens']) for u in usages),
            'num_tokens': max(int(u['num_tokens']) for u in usages),
            'num_requests': max(int(u['num_requests']) for u in usages),
        }

    max_single_price = price(max_usage([
        usage for usages in example_to_parsed_usage.values() 
        for usage in usages
    ]))

    # Calculate uncorrected prices
    example_to_price = {
        example_name: price(usage) 
        for example_name, usage in example_to_usage.items()
    }

    successful_example_to_price = {
        example_name: price(usage)
        for example_name, usage in example_to_usage.items() 
        if example_name in successes
    }

    # Calculate corrected prices
    corrected_example_to_price = example_to_price.copy()
    corrected_successful_example_to_price = successful_example_to_price.copy()

    # Add corrections for examples without messages
    examples_with_messages = set(example_to_price.keys())
    examples_without_messages = successes - examples_with_messages

    for example_name in examples_without_messages:
        if example_name in hammer_successes:
            # Hammer successes cost nothing
            corrected_example_to_price[example_name] = 0
            if example_name in successes:
                corrected_successful_example_to_price[example_name] = 0
        elif example_name in zero_shot_successes:
            # Zero-shot successes cost max_single_price
            corrected_example_to_price[example_name] = max_single_price
            if example_name in successes:
                corrected_successful_example_to_price[example_name] = max_single_price
        else:
            # warn that we are excluding this example
            print(f"WARNING: {example_name} has no messages, but is not a hammer or zero-shot success")

    return {
        "uncorrected_all": stats(example_to_price),
        "uncorrected_successful": stats(successful_example_to_price),
        "corrected_all": stats(corrected_example_to_price),
        "corrected_successful": stats(corrected_successful_example_to_price)
    }

In [ ]:
example_to_parsed_usage, example_to_usage = calculate_example_to_usage(ARGS[0])
usage_stats = calculate_usage_stats(example_to_parsed_usage, example_to_usage)

print("USAGE STATS")
print(json.dumps(usage_stats, indent=2))

print("COSTS")
print(json.dumps(calculate_cost_stats(example_to_parsed_usage, example_to_usage, *ARGS[1:]), indent=2))

## time

In [ ]:
from datetime import datetime

def is_file_prefix_command(line: t.Dict[str, str]) -> bool:
    return line['message'].startswith('running file prefix command:')

def compute_duration(earliest: datetime, latest: datetime, correction_s: float) -> float:
    return (latest - earliest).total_seconds() - correction_s

In [ ]:
def calculate_time_stats(
    log_path: Path,
    successes: t.Set[str],
    hammer_successes: t.Set[str],
    zero_shot_successes: t.Set[str],
    default_time_seconds: float = 30.0
) -> t.Dict[str, Stats]:
    """
    Calculate time statistics from log data.
    
    Args:
        log_path: Path to the log file
        successes: Set of successful example names
        hammer_successes: Set of examples solved by hammer
        zero_shot_successes: Set of examples solved by zero-shot
        default_time_seconds: Default time to use for examples without messages
        
    Returns:
        Dictionary containing stats for different time metrics
    """
    # Parse log file
    with open(log_path, 'r') as f:
        lines = f.readlines()
    lines_dicts = list(map(lambda l: json.loads(l), lines))

    # Calculate timestamps per example
    example_earliest_and_latest_timestamps = {}
    for line in lines_dicts:
        if 'example_name' not in line:
            continue
        example_name = line['example_name']
        timestamp = datetime.strptime(line['timestamp'], '%Y-%m-%d %H:%M:%S')
        
        if example_name not in example_earliest_and_latest_timestamps:
            example_earliest_and_latest_timestamps[example_name] = {
                'earliest': timestamp,
                'latest': timestamp,
                'last_line': line,
                'correction_s': 0,
                'file_prefix_correction_s': 0
            }
        else:
            prev_latest = example_earliest_and_latest_timestamps[example_name]['latest']
            # Add correction for gaps > 10 minutes
            if (timestamp - prev_latest).total_seconds() > 10 * 60:
                example_earliest_and_latest_timestamps[example_name]['correction_s'] += (timestamp - prev_latest).total_seconds()
            # Add correction for file prefix commands
            elif is_file_prefix_command(line) or is_file_prefix_command(example_earliest_and_latest_timestamps[example_name]['last_line']):
                example_earliest_and_latest_timestamps[example_name]['file_prefix_correction_s'] += (timestamp - prev_latest).total_seconds()
            example_earliest_and_latest_timestamps[example_name]['latest'] = timestamp
            example_earliest_and_latest_timestamps[example_name]['last_line'] = line

    def compute_duration(earliest: datetime, latest: datetime, correction_s: float) -> float:
        return (latest - earliest).total_seconds() - correction_s

    # Calculate durations with and without file prefix time
    example_duration_seconds = {
        example_name: compute_duration(dict['earliest'], dict['latest'], dict['correction_s'])
        for example_name, dict in example_earliest_and_latest_timestamps.items()
    }

    successful_example_duration_seconds = {
        example_name: duration
        for example_name, duration in example_duration_seconds.items()
        if example_name in successes
    }

    example_duration_seconds_no_file_prefix = {
        example_name: compute_duration(dict['earliest'], dict['latest'], dict['correction_s'] + dict['file_prefix_correction_s'])
        for example_name, dict in example_earliest_and_latest_timestamps.items()
    }

    successful_example_duration_seconds_no_file_prefix = {
        example_name: duration
        for example_name, duration in example_duration_seconds_no_file_prefix.items()
        if example_name in successes
    }

    # Add default times for examples without messages
    examples_with_messages = set(example_duration_seconds.keys())
    examples_without_messages = successes - examples_with_messages

    for example_name in examples_without_messages:
        if example_name in hammer_successes or example_name in zero_shot_successes:
            example_duration_seconds[example_name] = default_time_seconds
            example_duration_seconds_no_file_prefix[example_name] = default_time_seconds
            if example_name in successes:
                successful_example_duration_seconds[example_name] = default_time_seconds
                successful_example_duration_seconds_no_file_prefix[example_name] = default_time_seconds
        else:
            # warn that we are excluding this example
            print(f"WARNING: {example_name} has no messages, but is not a hammer or zero-shot success")

    return {
        "all": stats(example_duration_seconds),
        "successful": stats(successful_example_duration_seconds),
        "all_no_file_prefix": stats(example_duration_seconds_no_file_prefix),
        "successful_no_file_prefix": stats(successful_example_duration_seconds_no_file_prefix)
    }

In [ ]:
print("TIME")
print(json.dumps(calculate_time_stats(*ARGS), indent=2))